In [0]:
!pip install langchain openai langchain-community PyPDF2 langchain-core

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 35.9 MB

In [0]:
dbutils.library.restartPython()

In [0]:
%sql
create volume if not exists  rag_databricks;

In [0]:
import os 
print(os.listdir('/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/'))

['New York Brochure.pdf', 'Margies Travel Company Info.pdf', 'London Brochure.pdf', 'Dubai Brochure.pdf', 'Las Vegas Brochure.pdf', 'San Francisco Brochure.pdf']


In [0]:
%sql 
create schema if not exists dbws01.rag_schema

In [0]:
#for loading the files from workspace dir, and picking the PDF files for chunking, also adding path and chunk ids
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader
import os 
documents=[]
data_dir = "/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/"

for file in os.listdir(data_dir):
  if file.endswith(".pdf"):
    path = os.path.join(data_dir, file)
    reader=PdfReader(path)
    text=""
    for page in reader.pages:
      text+=page.extract_text()
    documents.append(Document(page_content=text, metadata={'path':path}))


text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=500,
    length_function=len
)

chunks = text_splitter.split_documents(documents)


all_docs = []
for i, chunk in enumerate(chunks):
    if chunk.page_content.strip():
        all_docs.append({
            'content_path': chunk.metadata['path'],
            'chunk': chunk.page_content,
            'chunk_id': i
        })

[{'content_path': '/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/New York Brochure.pdf',
  'chunk': 'Margie’s Travel Presents…  \nNew York  \nThe City of New York, often called New York City or simply \nNew York, is the most populous city in the United States. \nWith an estimated 2015 population of 8,550,405 distributed \nover a land area of about 302.6 square miles, New York \nCity is also the most densely populated major city in the \nUnited States. Located at the southern tip of the state of \nNew York, the city is the center of the New York \nmetropolitan area, one of the most populous urban \nagglomerati ons in the world. A global power city, New York \nCity exerts a significant impact upon commerce, finance, \nmedia, art, fashion, research, technology, education, and \nentertainment, its fast pace defining the term New York \nminute. Home to the headquarters of t he United Nations, \nNew York is an important center for international diplomacy \nand has been described 

In [0]:
df=spark.createDataFrame(all_docs)
print(f"Total chunks created: {df.count()}")

Total chunks created: 6


In [0]:
display(df)

chunk,chunk_id,content_path
"Margie’s Travel Presents… New York The City of New York, often called New York City or simply New York, is the most populous city in the United States. With an estimated 2015 population of 8,550,405 distributed over a land area of about 302.6 square miles, New York City is also the most densely populated major city in the United States. Located at the southern tip of the state of New York, the city is the center of the New York metropolitan area, one of the most populous urban agglomerati ons in the world. A global power city, New York City exerts a significant impact upon commerce, finance, media, art, fashion, research, technology, education, and entertainment, its fast pace defining the term New York minute. Home to the headquarters of t he United Nations, New York is an important center for international diplomacy and has been described as the cultural and financial capital of the world . New York Hotels Margie’s Travel offers the following accommodation options in New York : The Manhattan Hotel Stay in the heart of New York City, within an easy walk to Times Square and Broadway . The Grand Central Hotel Comfortable mid -town hotel . Close to Grand Central Station, the Chrysler Building, and the Empire State Building. The Park Hotel Luxurious accommodation in upper Manhattan, with views of Central Park . To book your trip to New York , visit www.margiestravel.com",0,/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/New York Brochure.pdf
"Wherever you want to go, Margie’s Travel can get you there ! Margie’s Travel is a world -leading travel agency, combining international reach with local expertise. Just tell us where you want to go, and we can arrange: • Flights • Accommodation • Transfers • Visas • Currency Exchange • Excursions Where We Go While we can arrange travel to anywhere worldwide, we specialize in trips to: • Dubai • Las Vegas • London • New York • San Francisco Who We Are Margie’s Travel employs some of the best travel experts in the world. Our leadership team consists of: • Marjorie Long (CEO) • Logan Reid (CFO) • Emma Luffman (CTO) • Deepak Nadar (Strategic Director) To learn more about us , visit our website at www.margiestrave l.com",1,/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/Margies Travel Company Info.pdf
"Margie’s Travel Presents… London London is the capital and most populous city of England and the United Kingdom. Standing on the River Thames in the south east of the island of Great Britain, London has been a major settlement for two millennia. It was founded by the Romans, who named it Londinium. London's ancient core, the City of London, largely retains its 1.12 -square - mile medieval boundaries. Since at least the 19th century, London has also referred to the metropolis around this core, historically split between Middlesex, Essex, Surre y, Kent, and Hertfordshire, which today largely makes up Greater London, governed by the Mayor of London and the London Assembly. Mostly popular for: Leisure, Outdoors, Historical, Arts & Culture Best time to visit: Jun-Aug Averag Precipitation : 1.9 in Average Temperature: 56-67°F London Hotels Margie’s Travel offers the following accommodation options in London : The Buckingham Hotel Comfortable hotel close to major sights like Buckingham Palace, Regent’s Park , and Trafalgar Square. The City Hotel Luxury rooms in the city, within walking distance of Tower Bridge and the Tower of London. . The Kensington Hotel Budget accommodation near Earl’s Court. To book your trip to London , visit www.margiestravel.com",2,/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/London Brochure.pdf
"Margie’s Travel Presents… Dubai Dubai is the largest and most populous city in the United Arab Emirates. It is located on the southeast coast of the Persian Gulf and is the capital of the Emirate of Dubai, one of the seven emirates that make up the country. Abu Dhabi and Dubai are the only two emirates to have veto power over c

In [0]:
df.write.mode('overwrite').saveAsTable('dbws01.rag_schema.RAG_docs_chunked')

In [0]:
%sql
select * from dbws01.rag_schema.rag_docs_chunked LIMIT 2;

chunk,chunk_id,content_path
"Margie’s Travel Presents… Las Vegas Las Vegas, officially the City of Las Vegas and often known simply as Vegas, is the 28th-most populated city in the United States, the most populated city in the state of Nevada, and the county seat of Clark County. The city anchors the Las Vegas Valley me tropolitan area and is the largest city within the greater Mojave Desert. Las Vegas is an internationally renowned major resort city, known primarily for its gambling, shopping, fine dining, entertainment, and nightlife. The Las Vegas Valley as a whole ser ves as the leading financial, commercial, and cultural center for Nevada. Las Vegas Hotels Margie’s Travel offers the following hotels in Las Vegas : The Volcano Hotel In the heart of The Strip . Stylish casino hotel with live entertainment and an extensive pool area. The Fountain Hotel Luxury accommodation in Las Vegas with a range of restaurants and cocktail bars The Canal Hotel An opulent Italian -themed resort with luxurious suite accommodation. To book your trip to Las Vegas , visit www.margiestravel.com",4,/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/Las Vegas Brochure.pdf
"Margie’s Travel Presents… San Francisco San Francisco, officially the City and County of San Francisco, is the cultural, commercial, and financial center of Northern California. It is the birthplace of the United Nations. Located at the north end of the San Francisco Peninsula, San Francisco is about 47.9 square miles in area, making it the smallest county —and the only consolidated city -county — within the state of California. With a density of about 18,581 people per square mile, San Franc isco is the most densely settled large city in California and the second -most densely populated major city in the United States after New York City. San Francisco is the fourth -most populous city in California, after Los Angeles, San Diego, and San Jose, a nd the 13th - most populous city in the United States —with a census -estimated 2016 population of 870,887. The city and its surrounding areas are known as the San Francisco Bay Area, and are a part of the larger OMB -designated San Jose -San Francisco -Oakland combined statistical area, the fifth most populous in the nation with an estimated population of 8.7 million. San Francisco Hotels Margie’s Travel offers the following accommodation options in San Francisco: The Lombard Hotel Friendly, family -run hotel within easy walking distance to popular sights, including the Golden Gate Bridge and the Presidio. Free Parking. The Wharf Hotel Great value accommodation in the heart of the vibrant Fisherman’s Whar f area . Continental Breakfast included. To book your trip to San Francisco, visit www.margiestravel.com",5,/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/San Francisco Brochure.pdf


In [0]:
#CDC Enable Change Data Capture, -keep a table that tracks snapshots
%pip install databricks-vectorsearch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#restart python kernel
dbutils.library.restartPython()

In [0]:
spark.sql(""" 
          ALTER TABLE rag_schema.rag_docs_chunked
          SET TBLPROPERTIES(delta.enableChangeDataFeed =true)
          """)

DataFrame[]

In [0]:
from databricks.vector_search.client import VectorSearchClient
vc=VectorSearchClient()

vc.create_endpoint(
    name='vectorsearch-endpoint',
    endpoint_type='STANDARD'
)


/root/.ipykernel/2003/command-6846049747980979-1952321374:1: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient
/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


---------------------------------------------------------------------------
BadRequest                                Traceback (most recent call last)
File <command-6846049747980979>, line 4
      1 from databricks.vector_search.client import VectorSearchClient
      2 vc=VectorSearchClient()
----> 4 vc.create_endpoint(
      5     name='vectorsearch-endpoint',
      6     endpoint_type='STANDARD'
      7 )

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-c8ab3d6f-bc88-4180-a244-26d4b47b10d9/lib/python3.12/site-packages/databricks/ai_search/client.py:310, in AISearchClient.create_endpoint(self, name, endpoint_type, budget_policy_id, usage_policy_id, target_qps)
    307 if target_qps is not None:
    308     json_data["target_qps"] = target_qps
--> 310 return self._issue_request(
    311     url=f"{self.workspace_url}/api/2.0/vector-search/endpoints",
    312     token=self._get_token_for_request(),
    313     method="POST",
    314     json=json_data,
    315 )

File /local_disk0/.ep

In [0]:

index=vc.create_delta_sync_index(
    endpoint_name='vectorsearch-endpoint',
    index_name='dbws01.rag_schema.ragindex',
    source_table_name='dbws01.rag_schema.rag_docs_chunked',
    primary_key='chunk_id',
    pipeline_type='TRIGGERED',
    embedding_source_column='chunk',
    embedding_model_endpoint_name='databricks-bge-large-en'

)

In [0]:
from databricks.vector_search.client import VectorSearchClient

vc=VectorSearchClient()
index=vc.get_index(
    endpoint_name='vectorsearch-endpoint',
    index_name='dbws01.rag_schema.ragindex')


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


index=vc.

In [0]:
#retrieve models
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
print([e.name for e in w.serving_endpoints.list()])


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
vector_question="can you tell me what hotels are offered by margies travel in dubai?"
result_dict=index.similarity_search(
    query_text=vector_question,
    num_results=10,
    query_type='hybrid' ,
    columns=['content_path', 'chunk']
)



[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
#retrieving results
import json 
for content in  result_dict['result']['data_array']:
    print(json.dumps(content, indent=4))

[
    "/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/Margies Travel Company Info.pdf",
    "Wherever you want to go, Margie\u2019s Travel can get you there ! \nMargie\u2019s Travel is a world -leading travel agency, combining international reach with local \nexpertise. Just tell us where you want to go, and we can arrange:  \n\u2022 Flights  \n\u2022 Accommodation  \n\u2022 Transfers  \u2022 Visas  \n\u2022 Currency Exchange  \n\u2022 Excursions  \n \nWhere We Go  \nWhile we can arrange travel to anywhere  \nworldwide, we specialize in trips to:  \n\u2022 Dubai  \n\u2022 Las Vegas  \n\u2022 London  \n\u2022 New York  \n\u2022 San Francisco  Who We Are  \nMargie\u2019s Travel employs some of the \nbest travel experts in the world. Our \nleadership team consists of:  \n\u2022 Marjorie  Long  (CEO)  \n\u2022 Logan  Reid  (CFO)  \n\u2022 Emma  Luffman  (CTO)  \n\u2022 Deepak  Nadar  (Strategic Director)  \n \nTo learn more about us , visit  our website at  www.margiestrave l.co

In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

DATABRICKS_TOKEN


'[REDACTED]'

In [0]:
%%writefile /Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/rag_model.py

import mlflow
from mlflow import pyfunc
from openai import OpenAI
from databricks.vector_search.client import VectorSearchClient

class RAGModel(pyfunc.PythonModel):
    #def __init__(self, vector_index):
    #    self.vector_index=vector_index
    def load_context(self, context):
        vc = VectorSearchClient(disable_notice=True)
        self.vector_index = vc.get_index(
            endpoint_name="vectorsearch-endpoint",
            index_name="dbws01.rag_schema.ragindex"
        )
    def retrieve(self, query):
        results_dict=self.vector_index.similarity_search(
            query_text=query,
            columns=['chunk_id','content_path', 'chunk'],
            num_results=10,
            query_type='hybrid'
        )
        return results_dict['result']['data_array']
    def chatCompletionsAPI(self, user_query, supporting_knowledge):
        client = OpenAI(
            api_key='dapid4078661620-3',
            base_url="https://adb-7405610296555497.17.azuredatabricks.net/ai-gateway/mlflow/v1"
        )

        completion=client.chat.completions.create(
            model='system.ai.gemma-3-12b',
            messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. You will be passed the user query and the supporting knowledge that can be used to answer the user query."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"user_query: {user_query}\nsupporting knowledge: {supporting_knowledge}"
                    }
            ]
        }]
        )
        return completion.choices[0].message.content

    def predict(self, context, model_input):
        query=model_input['user_query'].iloc[0]
        text_data=self.retrieve(query)
        return self.chatCompletionsAPI(query, text_data)
mlflow.models.set_model(RAGModel())




Overwriting /Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/rag_model.py


In [0]:
index.name

'dbws01.rag_schema.ragindex'

In [0]:
from mlflow.models import infer_signature
import pandas as pd 
input_example=pd.DataFrame(
    [
        {'user_query': "how is Gotham city?"}
    ]
)

output_example=pd.DataFrame(
    [
        {
        'predictions': 'Gotham city is facing cloudy weather'
        }
    ]
)
signature=infer_signature(input_example, output_example)



In [0]:
mlflow.pyfunc.save_model(path=model_path, python_model=test_model, signature=signature)

In [0]:
%pip install databricks-vectorsearch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
from databricks.vector_search.client import VectorSearchClient

from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)
resources = [
    DatabricksServingEndpoint(endpoint_name="databricks-claude-haiku-5-5"),
    DatabricksVectorSearchIndex(index_name="dbws01.rag_schema.ragindex"),
]

mlflow.pyfunc.log_model(
    python_model="/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/rag_model.py",
    artifact_path="agent",
    resources=resources,
    input_example=input_example,
    signature=signature,
    registered_model_name='rag_model'
)

2026/10/09 21:32:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405610296555497.17.azuredatabricks.net/ml/experiments/3877438775883370/models/m-025bf0694ad146c5981b2048b5f5a6cf?o=7405610296555497
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c8ab3d6f-bc88-4180-a244-26d4b47b10d9/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/09 21:32:14 INFO mlflow.pyfunc: Validating input example against model signature
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c8ab3d6f-bc88-4180-a244-26d4b47b10d9/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data v

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'rag_model' already exists. Creating a new version of this model...
2026/10/09 21:32:34 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to fetch model version's source run (with ID ) from tracking server. The source run may be deleted or inaccessible to the current user. No run link will be recorded for the model version.
2026/10/09 21:32:34 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to get model version source run's workspace ID from request headers. No run link will be recorded for the model version


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '8' of model 'dbws01.default.rag_model': https://adb-7405610296555497.17.azuredatabricks.net/explore/data/models/dbws01/default/rag_model/version/8?o=7405610296555497


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

w = WorkspaceClient()

#w.serving_endpoints.create(
#    name="rag-modelsb",
#    config=EndpointCoreConfigInput(
#        served_entities=[
#            ServedEntityInput(
#                entity_name="dbws01.default.rag_model",
#                entity_version="3",
#                workload_size="Small",
#                scale_to_zero_enabled=True,
#            )
#        ]
#    ),
#)

w.serving_endpoints.update_config(
    name="rag-modelsb",
    served_entities=[
        ServedEntityInput(
            entity_name="dbws01.default.rag_model",
            entity_version="8",
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ],
)

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = 'https://adb-7405610296555497.17.azuredatabricks.net/serving-endpoints/rag-modelsb/invocations'
    headers = {'Authorization': f'Bearer {os.environ.get("DATABRICKS_TOKEN")}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()
    

---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-4794005219913127>, line 24
     21 os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
     23 dataset = pd.DataFrame([{"user_query": "What does the Dubai brochure say about payment plans?"}])
---> 24 print(score_model(dataset))

File <command-4794005219913127>, line 18, in score_model(dataset)
     16 response = requests.request(method='POST', headers=headers, url=url, data=data_json)
     17 if response.status_code != 200:
---> 18     raise Exception(f'Request failed with status {response.status_code}, {response.text}')
     19 return response.json()

Exception: Request failed with status 400, {"error_code": "BAD_REQUEST", "message": "Encountered an unexpected error while evaluating the model. Verify that the input is compatible with the model for inference.

In [0]:
#test_model=RAGModel(vector_index='dbws01.rag_schema.ragindex')
test_model=RAGModel()

In [0]:
#logging model as artifact
run_id=None
with mlflow.start_run() as run:
    mlflow.log_artifact(model_path)
    run_id=run.info.run_id
    print("Model logged",run_id)

Model logged 97eeb59944054cc085f1ec88b97c3f7c


In [0]:
#registering model within unity catalog
mlflow.register_model(f"runs:/{run_id}/{model_path}","rag_model")

Successfully registered model 'dbws01.default.rag_model'.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

🔗 Created version '1' of model 'dbws01.default.rag_model': https://adb-7405610296555497.17.azuredatabricks.net/explore/data/models/dbws01/default/rag_model/version/1?o=7405610296555497


<ModelVersion: aliases=[], creation_timestamp=1791574394919, current_stage=None, deployment_job_state=None, description='', last_updated_timestamp=1791574396872, metrics=[], model_id='', name='dbws01.default.rag_model', params=[], run_id='97eeb59944054cc085f1ec88b97c3f7c', run_link=None, source='dbfs:/databricks/mlflow-tracking/3877438775883370/97eeb59944054cc085f1ec88b97c3f7c/artifacts/rag_model2', status='READY', status_message='', tags={}, user_id='satyakeb@omnics1234.onmicrosoft.com', version='1', workspace='default'>

In [0]:
{
  "dataframe_split": {
    "columns": ["user_query"],
    "data": [["What does the Dubai brochure say about payment plans?"]]
  }
}